# <div style='background-color: #1e3c72; color: white; padding: 20px; border-radius: 10px; text-align: center;'>🛰️ Manipulation Experte de Données MTG</div>
### Workflow complet : GDAL & ImageMagick (Bash)

## ⚙️ Initialisation de l'environnement

In [1]:
!mkdir -p RESULTS
IN="/toto/tata/titi/mon_image.tif"
print(f"Fichier source configuré : {IN}")

Fichier source configuré : /toto/tata/titi/mon_image.tif


## 1. Inspection des données

In [ ]:
!gdalinfo $IN
!identify $IN

## 2. Reprojection & Découpe (France)
Utilisation de `gdalwarp` avec options d'optimisation.

In [ ]:
# Reprojection en 4326 + Découpe France + Redimensionnement
!gdalwarp -t_srs EPSG:4326 -te -5.5 41.0 10.0 51.5 -ts 2400 0 -r bilinear $IN RESULTS/france_4326.tif

# Miniature
!convert RESULTS/france_4326.tif -resize 500x RESULTS/thumb_france.jpg
from IPython.display import Image
Image(filename='RESULTS/thumb_france.jpg')

## 3. Amélioration Image (Gamma & Exponent)
Correction de la dynamique pour un rendu visuel optimal.

In [ ]:
# Application d'un exposant pour corriger la luminosité
!gdal_translate -exponent 0.6 RESULTS/france_4326.tif RESULTS/france_gamma.tif

# Miniature
!convert RESULTS/france_gamma.tif -resize 500x RESULTS/thumb_gamma.jpg
Image(filename='RESULTS/thumb_gamma.jpg')

## 4. Extraction de valeurs ponctuelles

In [ ]:
# Récupérer les valeurs RGB au centre de la France
!gdallocationinfo -wgs84 RESULTS/france_4326.tif 1.5 46.5

## 5. Annotations & Cosmétique (Vecteur + Texte)

In [ ]:
# Ajout du trait de côte (en jaune)
!gdal_rasterize -b 1 -b 2 -b 3 -burn 255 255 0 -l coastlines coastlines.shp RESULTS/france_gamma.tif

# Ajout de texte avec ImageMagick
!convert RESULTS/france_gamma.tif -gravity NorthWest -pointsize 40 -fill white \
         -annotate +50+50 "SATELLITE MTG - FRANCE" RESULTS/PRODUIT_FINAL.tif

# Miniature finale
!convert RESULTS/PRODUIT_FINAL.tif -resize 500x RESULTS/final.jpg
Image(filename='RESULTS/final.jpg')